
# UPSkill 1S2627 — Companies You Know, Data You Can Use

### Part B :: Colab Camp — filter a shortlist from one file (no install)

**Educational exercise · not investment advice · figures rounded and dated · not affiliated with PSE**

What you will do, in order:

1. Load one offline snapshot: 20 familiar PH companies (`demo_company_metrics.csv`)
2. Peek at the table and count the blanks
3. Keep only complete rows, sort by dividend yield, then change the rule with a partner

Run every cell with **Shift + Enter**. Read the output before moving on. Ask anytime.


## 01 :: Setup

Colab already has `pandas` and `matplotlib`. Nothing to install.

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

print("pandas", pd.__version__)


## 02 :: Load the data

The snapshot is frozen: 20 rows, 9 columns. `last_price` and `div_yield_pct` are precomputed — nobody divides anything today.

In [ ]:

# The same line from the slides, pointed at the course copy of the file.
# It tries `main` first, then the workshop branch that holds the file today.
# skipinitialspace trims stray spaces so a blank cell counts as missing.
URLS = [
    "https://raw.githubusercontent.com/SenjoNanaya/pse-dividend-analysis/main/workshop/upskill-1s2627/resources/demo_company_metrics.csv",
    "https://raw.githubusercontent.com/SenjoNanaya/pse-dividend-analysis/cursor/upskill-1s2627-workshop-proposal-ae07/workshop/upskill-1s2627/resources/demo_company_metrics.csv",
]

df = None
for url in URLS:
    try:
        df = pd.read_csv(url, skipinitialspace=True)
        print("Loaded", len(df), "rows")
        break
    except Exception:
        pass

if df is None:
    # Offline fallback: upload demo_company_metrics.csv to this Colab session, then rerun.
    df = pd.read_csv("demo_company_metrics.csv", skipinitialspace=True)

df.shape  # expect (20, 9)


## 03 :: Peek at the table

`familiar_as` is the name you know, `ticker` is the shorthand, `data_complete` flags the row.

In [ ]:

df.head()

In [ ]:

df.shape  # (20, 9)


## 04 :: Count the blanks

`isna()` counts missing cells. A blank yield is missing — never zero. `BLOOM 0.0` is a real zero and stays.

`debt_to_equity` also shows two blanks, for BDO and BPI. Those are deliberate — banks skip industrial debt math.

In [ ]:

df.isna().sum()

In [ ]:

# Rows with a missing yield. Expect MPI, CNVRG, RRHI.
df[df["div_yield_pct"].isna()][["ticker", "familiar_as", "div_yield_pct", "data_complete"]]


## 05 :: The speaker default filter

Keep `data_complete == "yes"`, drop the blank yields, then sort largest first. 20 rows become 17 before the sort.

In [ ]:

complete = df[df.data_complete == "yes"]
complete = complete.dropna(subset=["div_yield_pct"])
short = complete.sort_values("div_yield_pct", ascending=False)

print("complete rows:", len(complete))  # expect 17
short[["ticker", "familiar_as", "last_price", "div_yield_pct"]]


## 06 :: Your turn — change the rule

One number only: the cutoff. Everything else stays frozen so screens stay comparable.

The name that tops the sort proves the sort works. It does not recommend the company.

In [ ]:

CUTOFF = 3.0   # <- change this number, then run the cell

above = short[short.div_yield_pct > CUTOFF]
print(f"{len(above)} companies with yield above {CUTOFF}%")
above[["ticker", "familiar_as", "div_yield_pct"]]


## 07 :: Chart the shortlist

Yield for up to 7 familiar names, complete rows only.

In [ ]:

top7 = short.head(7).iloc[::-1]   # reversed so the largest bar sits on top

ax = top7.plot.barh(
    x="familiar_as",
    y="div_yield_pct",
    legend=False,
    color="#4E4A3E",
    figsize=(8, 4),
)
ax.set_xlabel("Dividend yield (% of price)")
ax.set_ylabel("")
ax.set_title("Dividend yield, complete rows only (offline demo snapshot)")
plt.tight_layout()
plt.show()


## 08 :: One sentence and one caveat

Say it out loud, then keep it for the share-out:

> Among complete rows, ______ because ______. Caveat: ______.

Counts as a caveat: a blank was dropped (not zero), a figure is rounded or dated, or a very high yield (SCC 12.0) needs a second look before anyone trusts it.


## Optional :: what the Guild does next

Today you did the loop by hand: ask, collect, clean, filter, caveat.
Later: fundamentals and ROIC, SQL and dashboards, the full EDGE scraping pipeline behind this snapshot — `github.com/SenjoNanaya/pse-dividend-analysis`.

---

*Educational snapshot, rounded and dated. No PSE affiliation. Not investment advice.*